# C5 — XGBoost Native Categorical (Kaggle GPU)

This notebook is the **Kaggle-GPU version** of our corrected native-categorical
XGBoost experiment.

### Model
XGBoost with:
- native categorical features
- `enable_categorical=True`
- `device="cuda"`
- histogram tree method
- fold-level early stopping

### Evaluation
Primary:
- PR-AUC ↑

Secondary:
- ROC-AUC ↑
- Log Loss ↓
- Brier Score ↓

### Leakage protocol
- C4 `train.csv` only for Optuna
- 5-fold `StratifiedGroupKFold`
- group = `case_id`
- validation/test untouched during tuning
- category vocabularies learned from TRAIN only


In [1]:
!nvidia-smi

Mon Aug 24 21:23:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import xgboost as xgb
import optuna

print("XGBoost:", xgb.__version__)
print("Optuna:", optuna.__version__)

XGBoost: 3.2.0
Optuna: 4.9.0


In [3]:

# ============================================================
# 0. IMPORTS + GPU CHECK
# ============================================================

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import joblib
import xgboost as xgb

from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style="whitegrid", context="notebook")

print("XGBoost version:", xgb.__version__)

version_parts = tuple(
    int(part) for part in xgb.__version__.split(".")[:2]
)
if version_parts < (2, 0):
    raise RuntimeError(
        "This Kaggle notebook requires XGBoost >= 2.0 "
        "for the current GPU/native-categorical configuration."
    )

try:
    import cupy as cp
    print("CuPy:", cp.__version__)
    print("CUDA device count:", cp.cuda.runtime.getDeviceCount())
except Exception as exc:
    print(
        "GPU sanity check warning:",
        exc,
        "\nThe XGBoost model will still attempt device='cuda'."
    )

SEED = 20260824
TARGET = "y"
GROUP_COL = "case_id"

N_SPLITS = 5
N_TRIALS = 50
MAX_ESTIMATORS = 1500
EARLY_STOPPING_ROUNDS = 60

REPORT_DIR = Path("/kaggle/working/reports/c5_xgb_native_categorical")
MODEL_DIR = Path("/kaggle/working/models/c5")
REPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)


XGBoost version: 3.2.0
CuPy: 14.0.1
CUDA device count: 2


In [4]:

# ============================================================
# KAGGLE INPUT DISCOVERY
# ============================================================
# This notebook is designed for Kaggle where train.csv,
# validation.csv, and test.csv may be attached as separate
# Kaggle datasets. We discover them automatically by filename.

from pathlib import Path
import pandas as pd

INPUT_ROOT = Path("/kaggle/input")

def find_unique_csv(filename: str) -> Path:
    matches = list(INPUT_ROOT.rglob(filename))
    if len(matches) == 0:
        raise FileNotFoundError(
            f"Could not find {filename} under {INPUT_ROOT}. "
            "Attach the corresponding Kaggle dataset."
        )
    if len(matches) > 1:
        raise RuntimeError(
            f"Found multiple copies of {filename}:\n"
            + "\n".join(str(p) for p in matches)
            + "\nPlease attach only one dataset containing this file."
        )
    return matches[0]

TRAIN_PATH = find_unique_csv("train.csv")
VALID_PATH = find_unique_csv("validation.csv")
TEST_PATH = find_unique_csv("test.csv")

print("TRAIN :", TRAIN_PATH)
print("VALID :", VALID_PATH)
print("TEST  :", TEST_PATH)

train = pd.read_csv(TRAIN_PATH, low_memory=False)
valid = pd.read_csv(VALID_PATH, low_memory=False)
test = pd.read_csv(TEST_PATH, low_memory=False)

print("\nShapes:")
print("Train      :", train.shape)
print("Validation :", valid.shape)
print("Test       :", test.shape)


TRAIN : /kaggle/input/datasets/sohamgaonkhadkar2/revenue-recovery-train/train.csv
VALID : /kaggle/input/datasets/sohamgaonkhadkar2/revenue-recovery-validation/validation.csv
TEST  : /kaggle/input/datasets/sohamgaonkhadkar2/revenue-recovery-test/test.csv

Shapes:
Train      : (110385, 37)
Validation : (23471, 37)
Test       : (23567, 37)


## 1. Feature Contract

In [5]:

ID_COLUMNS = {
    "customer_id",
    "subscription_id",
    "case_id",
}

POLICY_COLUMNS = {
    "logging_policy",
}

INTERVENTION_COLUMNS = {
    "action_type",
    "channel",
}

FEATURES = [
    c for c in train.columns
    if c not in ID_COLUMNS | POLICY_COLUMNS | {TARGET}
]

CATEGORICAL_COLS = [
    c for c in FEATURES
    if (
        c in INTERVENTION_COLUMNS
        or not pd.api.types.is_numeric_dtype(train[c])
    )
]

NUMERIC_COLS = [
    c for c in FEATURES
    if c not in CATEGORICAL_COLS
]

assert not (set(FEATURES) & ID_COLUMNS)
assert not (set(FEATURES) & POLICY_COLUMNS)

print("Predictive features:", len(FEATURES))
print("Numeric:", len(NUMERIC_COLS))
print("Categorical:", len(CATEGORICAL_COLS))
print(CATEGORICAL_COLS)


Predictive features: 32
Numeric: 20
Categorical: 12
['rail', 'bank_name', 'segment', 'plan_tier', 'value_tier', 'first_failure_class', 'failure_class_at_decision', 'age_band', 'state', 'locale_pref', 'action_type', 'channel']


## 2. Native Categorical Preparation

In [6]:

# Learn category vocabularies from TRAIN only.
category_vocab = {}

for col in CATEGORICAL_COLS:
    values = (
        train[col]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    if "__UNKNOWN__" not in values:
        values.append("__UNKNOWN__")

    category_vocab[col] = values


def prepare_native_frame(df):
    X = df[FEATURES].copy()

    for col in CATEGORICAL_COLS:
        vocab = category_vocab[col]

        values = (
            X[col]
            .astype("string")
            .fillna("__UNKNOWN__")
            .astype(str)
        )

        values = values.where(
            values.isin(vocab),
            "__UNKNOWN__",
        )

        X[col] = pd.Categorical(
            values,
            categories=vocab,
        )

    return X


X_train = prepare_native_frame(train)
X_valid = prepare_native_frame(valid)
X_test = prepare_native_frame(test)

y_train = train[TARGET].astype(int)
y_valid = valid[TARGET].astype(int)
y_test = test[TARGET].astype(int)
groups = train[GROUP_COL]

print("Prepared native categorical frames.")


Prepared native categorical frames.


## 3. Evaluation Metrics

In [7]:

def evaluate(y_true, proba):
    proba = np.clip(
        np.asarray(proba, dtype=float),
        1e-7,
        1 - 1e-7,
    )
    y_true = np.asarray(y_true)

    return {
        "pr_auc": float(
            average_precision_score(y_true, proba)
        ),
        "roc_auc": float(
            roc_auc_score(y_true, proba)
        ),
        "log_loss": float(
            log_loss(y_true, proba)
        ),
        "brier": float(
            brier_score_loss(y_true, proba)
        ),
    }


## 4. Optuna Search Space

In [8]:

def xgb_params(trial):
    return {
        "max_depth": trial.suggest_int(
            "max_depth", 3, 10
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.008, 0.08, log=True
        ),
        "min_child_weight": trial.suggest_int(
            "min_child_weight", 1, 40
        ),
        "subsample": trial.suggest_float(
            "subsample", 0.70, 1.00
        ),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.70, 1.00
        ),
        "gamma": trial.suggest_float(
            "gamma", 0.0, 3.0
        ),
        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-5, 5.0, log=True
        ),
        "reg_lambda": trial.suggest_float(
            "reg_lambda", 0.05, 30.0, log=True
        ),
        "max_cat_to_onehot": trial.suggest_int(
            "max_cat_to_onehot", 1, 8
        ),
        "max_cat_threshold": trial.suggest_int(
            "max_cat_threshold", 32, 128, step=16
        ),
    }


## 5. Case-Grouped CV Objective

In [9]:

cv = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)

def xgb_objective(trial):
    params = xgb_params(trial)

    fold_metrics = []
    best_iterations = []

    for fold, (tr_idx, va_idx) in enumerate(
        cv.split(
            X_train,
            y_train,
            groups,
        )
    ):
        X_tr = X_train.iloc[tr_idx]
        X_va = X_train.iloc[va_idx]

        y_tr = y_train.iloc[tr_idx]
        y_va = y_train.iloc[va_idx]

        model = XGBClassifier(
            **params,
            n_estimators=1500,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            enable_categorical=True,
            device="cuda",
            early_stopping_rounds=60,
            random_state=SEED + fold,
            n_jobs=-1,
        )

        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_va, y_va)],
            verbose=False,
        )

        best_iterations.append(
            int(
                getattr(
                    model,
                    "best_iteration",
                    MAX_ESTIMATORS - 1,
                )
            ) + 1
        )

        proba = model.predict_proba(
            X_va
        )[:, 1]

        fold_metrics.append(
            evaluate(
                y_va,
                proba,
            )
        )

    mean_metrics = {
        metric: float(
            np.mean(
                [
                    result[metric]
                    for result in fold_metrics
                ]
            )
        )
        for metric in [
            "pr_auc",
            "roc_auc",
            "log_loss",
            "brier",
        ]
    }

    for metric, value in mean_metrics.items():
        trial.set_user_attr(
            f"cv_{metric}",
            value,
        )

    trial.set_user_attr(
        "median_best_iteration",
        int(np.median(best_iterations)),
    )

    return mean_metrics["pr_auc"]


## 6. Run Optuna

In [10]:

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        seed=SEED,
    ),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=6,
        n_warmup_steps=2,
    ),
)

study.optimize(
    xgb_objective,
    n_trials=N_TRIALS,
    show_progress_bar=True,
)

best_trial = study.best_trial

print("Completed trials:", len(study.trials))
print("Best CV PR-AUC:", study.best_value)
print("Best CV ROC-AUC:", best_trial.user_attrs["cv_roc_auc"])
print("Best CV Log Loss:", best_trial.user_attrs["cv_log_loss"])
print("Best CV Brier:", best_trial.user_attrs["cv_brier"])
print(
    "Median best iteration:",
    best_trial.user_attrs["median_best_iteration"],
)
print("\nBest parameters:")
print(best_trial.params)


  0%|          | 0/50 [00:00<?, ?it/s]

Completed trials: 50
Best CV PR-AUC: 0.8124808446565689
Best CV ROC-AUC: 0.9330365199016555
Best CV Log Loss: 0.26682309648893454
Best CV Brier: 0.08316848000898618
Median best iteration: 1256

Best parameters:
{'max_depth': 9, 'learning_rate': 0.008963811131925235, 'min_child_weight': 1, 'subsample': 0.7610778573531095, 'colsample_bytree': 0.7335192299649966, 'gamma': 2.314635502713329, 'reg_alpha': 0.0045876353408079934, 'reg_lambda': 0.8634870399650165, 'max_cat_to_onehot': 7, 'max_cat_threshold': 48}


## 7. Final Native-Categorical XGBoost

In [11]:

selected_n_estimators = int(
    best_trial.user_attrs["median_best_iteration"]
)

final_model = XGBClassifier(
    **best_trial.params,
    n_estimators=selected_n_estimators,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    enable_categorical=True,
    device="cuda",
    random_state=SEED,
    n_jobs=-1,
)

final_model.fit(
    X_train,
    y_train,
    verbose=False,
)

train_proba = final_model.predict_proba(
    X_train
)[:, 1]
valid_proba = final_model.predict_proba(
    X_valid
)[:, 1]
test_proba = final_model.predict_proba(
    X_test
)[:, 1]

native_results = pd.DataFrame([
    {
        "model": "xgboost_native_categorical",
        "split": "train",
        **evaluate(y_train, train_proba),
    },
    {
        "model": "xgboost_native_categorical",
        "split": "validation",
        **evaluate(y_valid, valid_proba),
    },
    {
        "model": "xgboost_native_categorical",
        "split": "test",
        **evaluate(y_test, test_proba),
    },
])

display(native_results)


,model,split,pr_auc,roc_auc,log_loss,brier
0,xgboost_native_categorical,train,0.936512,0.978322,0.182110,0.052692
1,xgboost_native_categorical,validation,0.812094,0.935437,0.264373,0.082482
2,xgboost_native_categorical,test,0.831340,0.940206,0.252191,0.077702


## 8. Save Results

In [12]:

native_results.to_csv(
    REPORT_DIR / "native_categorical_metrics.csv",
    index=False,
)

study.trials_dataframe().to_csv(
    REPORT_DIR / "optuna_trials.csv",
    index=False,
)

with open(
    REPORT_DIR / "best_params.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        {
            "model": "xgboost_native_categorical",
            "optimization_metric": "pr_auc",
            "cv": "5-fold StratifiedGroupKFold",
            "group": GROUP_COL,
            "best_cv_pr_auc": float(study.best_value),
            "best_cv_roc_auc":
                best_trial.user_attrs["cv_roc_auc"],
            "best_cv_log_loss":
                best_trial.user_attrs["cv_log_loss"],
            "best_cv_brier":
                best_trial.user_attrs["cv_brier"],
            "n_estimators": selected_n_estimators,
            "best_params": best_trial.params,
        },
        f,
        indent=2,
    )

# JSON is used because native categorical metadata must be preserved.
final_model.save_model(
    MODEL_DIR / "xgboost_native_categorical_c5.json"
)

joblib.dump(
    {
        "features": FEATURES,
        "categorical_features": CATEGORICAL_COLS,
        "category_vocab": category_vocab,
    },
    REPORT_DIR / "native_categorical_schema.joblib",
)

print("Saved outputs to:", REPORT_DIR)
print(
    "Model:",
    MODEL_DIR / "xgboost_native_categorical_c5.json",
)


Saved outputs to: /kaggle/working/reports/c5_xgb_native_categorical
Model: /kaggle/working/models/c5/xgboost_native_categorical_c5.json
